# Imports

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import OxfordIIITPet
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import classification_report
import random
from collections import defaultdict
import copy
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device: ' + str(device))

Using device: cuda


In [ ]:
#Returns indices for split of training data with fraction validation data and (1 - fraction) training data
def stratified_train_val_indices(training_set, val_frac, seed=42):
    random.seed(seed)
    class_to_indices = defaultdict(list)

    for idx in range(len(training_set)):
        label = training_set[idx][1]
        class_to_indices[label].append(idx)

    train_data_indices = []
    val_data_indices = []

    for label, indices in class_to_indices.items():
        random.shuffle(indices)
        n_val = max(1, int(len(indices) * val_frac))
        val_data_indices.extend(indices[:n_val]) #adds first n_keep indices
        train_data_indices.extend(indices[n_val:]) #adds all indices after first n_keep indices => different indices (images) than the validation data

    random.shuffle(val_data_indices)
    random.shuffle(train_data_indices)

    return train_data_indices, val_data_indices

# Data

In [ ]:
# Since params inside ResNet18 is based of these values, we should use them and not the ones for oxford-IIIT
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

# Again 224 is needed since its what ResNet18 wants as input
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

# Data augmentation transform
train_transform_aug = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), #crops + small size scaling (80% - 100% of original image)
    transforms.RandomHorizontalFlip(p=0.5),  #flip
    transforms.RandomRotation(15),               #small rotations (between -15 and 15 degrees)
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

# For binary (0,1)
all_train_set_binary = OxfordIIITPet(root='data', split='trainval', target_types='binary-category', transform=train_transform, download=True)
test_set_binary  = OxfordIIITPet(root='data', split='test', target_types='binary-category', transform=test_transform,  download=True)

val_frac = 0.1 #10% validation data
train_data_indices, val_data_indices = stratified_train_val_indices(all_train_set_binary, val_frac, seed=42)
train_set_binary = Subset(all_train_set_binary, train_data_indices)
val_set_binary = Subset(all_train_set_binary, val_data_indices)

all_train_loader_binary = DataLoader(all_train_set_binary, batch_size=50, shuffle=True,  num_workers=0) # Maybe batch size or/and num_workers should change value
train_loader_binary = DataLoader(train_set_binary, batch_size=50, shuffle=True,  num_workers=0) # Maybe batch size or/and num_workers should change value
val_loader_binary  = DataLoader(val_set_binary, batch_size=50, shuffle=False, num_workers=0)
test_loader_binary  = DataLoader(test_set_binary, batch_size=50, shuffle=False, num_workers=0)

# For full 37 (0-36)
all_train_set_full = OxfordIIITPet(root='data', split='trainval', target_types='category', transform=train_transform, download=True)
test_set_full  = OxfordIIITPet(root='data', split='test', target_types='category', transform=test_transform,  download=True)
all_train_set_full_aug = OxfordIIITPet(root='data', split='trainval', target_types='category', transform=train_transform_aug, download=True)

val_frac = 0.1 #10% validation data
train_data_indices, val_data_indices = stratified_train_val_indices(all_train_set_full, val_frac, seed=42)
train_set_full = Subset(all_train_set_full, train_data_indices)
val_set_full = Subset(all_train_set_full, val_data_indices)
train_set_full_aug = Subset(all_train_set_full_aug, train_data_indices)


all_train_loader_full = DataLoader(all_train_set_full, batch_size=50, shuffle=True,  num_workers=0)
train_loader_full = DataLoader(train_set_full, batch_size=50, shuffle=True,  num_workers=0)
val_loader_full = DataLoader(val_set_full, batch_size=50, shuffle=False,  num_workers=0)
test_loader_full  = DataLoader(test_set_full, batch_size=50, shuffle=False, num_workers=0)
train_loader_full_aug = DataLoader(train_set_full_aug, batch_size=50, shuffle=True,  num_workers=0)

print(f'Size of all training data {len(all_train_set_full)}')
print(f'Train size for binary: {len(train_set_binary)}, Validation size: {len(val_set_binary)} Test size: {len(test_set_binary)}')
print(f'Train size for full: {len(train_set_full)}, Validation size: {len(val_set_full)}, Test size: {len(test_set_full)}')
print(f'Train size for full aug: {len(train_set_full_aug)}, Validation size: {len(val_set_full)}, Test size: {len(test_set_full)}')

100%|██████████| 792M/792M [00:03<00:00, 230MB/s]
100%|██████████| 19.2M/19.2M [00:00<00:00, 91.3MB/s]


Size of all training data 3680
Train size for binary: 3313, Validation size: 367 Test size: 3669
Train size for full: 3315, Validation size: 365, Test size: 3669
Train size for full aug: 3315, Validation size: 365, Test size: 3669


# Training for Binary Classification

In [ ]:
model = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device) # Only choice (of weights param)? # Added .to(device) to alow for T4 GPU, CPU still works /Björn

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2).to(device)
model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 187MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
# Training for binary classification
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=5e-5) # Maybe lr should change for greater than 99%

num_epochs = 4

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in all_train_loader_binary:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(all_train_set_binary)
    train_acc  = correct / len(all_train_set_binary)


    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader_binary:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # 0 = cat, 1 = dog for binary
            correct += (outputs.argmax(1) == labels).sum().item()

    test_acc = correct / len(test_set_binary)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')

Epoch [01/4]  Loss: 0.0256  Train Acc: 0.9913  Test Acc: 0.9877
Epoch [02/4]  Loss: 0.0225  Train Acc: 0.9908  Test Acc: 0.9880
Epoch [03/4]  Loss: 0.0237  Train Acc: 0.9918  Test Acc: 0.9888
Epoch [04/4]  Loss: 0.0302  Train Acc: 0.9910  Test Acc: 0.9886


In [ ]:
print('Final test accuracy for binary classification: ' + str(test_acc*100) + '%') # I get 97.7% after 2 epochs in vscode (~3min per epoch, takes a lot longer in colab)

Final test accuracy for binary classification: 98.85527391659853%


# Training for Full Classification

In [ ]:
model = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device) # Only choice (of weights param)?

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 37).to(device)
model.to(device)

In [ ]:
# Training for full classification
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_full:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_full)
    train_acc  = correct / len(train_set_full)


    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in val_loader_full: #changed from test to val
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # 0-36 depending on breed
            correct += (outputs.argmax(1) == labels).sum().item()

    val_acc = correct / len(val_set_full) #changed from test to val

    print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Val Acc: {val_acc:.4f}')

In [ ]:
print('Final validation accuracy for full classification: ' + str(val_acc*100) + '%') # I get 84.4% after 2 epochs in vscode (~3min per epoch, takes a lot longer in colab)

In [ ]:
print(f'Final val accuracy for full classification for different learning rates')

lrs = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
for lr in lrs:
  model = modelGeneration()
  val_acc = fullClassification(model, train_loader_full, lr)
  print(f'Final val accuracy for full classification for learning rate {lr}: {val_acc*100:.3f}%')

"""Final val accuracy for full classification for learning rate 0.0001: 33.973%
Final val accuracy for full classification for learning rate 0.0005: 82.740%
Final val accuracy for full classification for learning rate 0.001: 86.849%
Final val accuracy for full classification for learning rate 0.005: 85.479%
Final val accuracy for full classification for learning rate 0.01: 85.753%"""

Final val accuracy for full classification for different learning rates
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 187MB/s]


Final val accuracy for full classification for learning rate 0.0001: 33.973%
Final val accuracy for full classification for learning rate 0.0005: 82.740%
Final val accuracy for full classification for learning rate 0.001: 86.849%
Final val accuracy for full classification for learning rate 0.005: 85.479%
Final val accuracy for full classification for learning rate 0.01: 85.753%


In [ ]:
print(f'Finer search: Final val accuracy for full classification for different learning rates')
lrs = [7e-4, 8e-4, 9e-4, 1e-3, 1.2e-3, 1.5e-3, 2e-3, 3e-3]
for lr in lrs:
  model = modelGeneration()
  val_acc = fullClassification(model, train_loader_full, lr)
  print(f'Final val accuracy for full classification for learning rate {lr}: {val_acc*100:.3f}%')

  """Finer search: Final val accuracy for full classification for different learning rates

Final val accuracy for full classification for learning rate 0.0007: 84.384%
Final val accuracy for full classification for learning rate 0.0008: 83.836%
Final val accuracy for full classification for learning rate 0.0009: 85.753%
Final val accuracy for full classification for learning rate 0.001: 84.658%
Final val accuracy for full classification for learning rate 0.0012: 87.671%
Final val accuracy for full classification for learning rate 0.0015: 87.671%
Final val accuracy for full classification for learning rate 0.002: 85.753%
Final val accuracy for full classification for learning rate 0.003: 89.863%

Best was: Final val accuracy for full classification for learning rate 0.003: 89.863%"""

Finer search: Final val accuracy for full classification for different learning rates
Final val accuracy for full classification for learning rate 0.0007: 84.384%




```
# This is formatted as code
```

# Strategy 1

Unfreeze layers

In [ ]:
"""model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

trainable_blocks = []

for name, module in model.named_children():
    has_params = any(p.requires_grad is not None for p in module.parameters())

    if has_params:
        trainable_blocks.append((name, module))

l = 3 # number of layers to unfreeze

last_blocks = trainable_blocks[-(l+1):-1]



for name, block in last_blocks:
    for param in block.parameters():
        param.requires_grad = True

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 37).to(device)
model = model.to(device)

"""

In [ ]:
"""# Training for full classification
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_full:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_full)
    train_acc  = correct / len(train_set_full)


    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader_full:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # 0-36 depending on breed
            correct += (outputs.argmax(1) == labels).sum().item()

    test_acc = correct / len(test_set_full)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')"""

In [ ]:
print('Final test accuracy for full classification with one unefreezed layer: ' + str(test_acc*100) + '%') # I get 86.9, 88.8, and 87,8% for 1, 2, 3 layers repsectively taking ~7, 8, 10 minutes

### Hyper parameter search for strategy 1

In [ ]:
numLays_values = [1, 2, 3]
lr_values = [1e-5, 5e-5, 1e-4, 5e-4]

print(f'Final val accuracy for full classification with strategy 1 for different learning rates and number of unfreeze layers')
for numLays in numLays_values:
    for lr in lr_values:
        val_acc =  modelGenFullClassStrat1(train_loader_full, lr ,numLays)
        print(f'Final val acc for full class for lr = {lr} and number of unfreeze layers = {numLays}: {val_acc*100:.3f}%')

"""Final val acc for full class for lr = 1e-05 and number of unfreeze layers = 1: 43.836%
Final val acc for full class for lr = 5e-05 and number of unfreeze layers = 1: 85.753%
Final val acc for full class for lr = 0.0001 and number of unfreeze layers = 1: 89.863%
Final val acc for full class for lr = 0.0005 and number of unfreeze layers = 1: 87.397%
Final val acc for full class for lr = 1e-05 and number of unfreeze layers = 2: 43.836%
Final val acc for full class for lr = 5e-05 and number of unfreeze layers = 2: 87.671%
Final val acc for full class for lr = 0.0001 and number of unfreeze layers = 2: 90.137%
Final val acc for full class for lr = 0.0005 and number of unfreeze layers = 2: 86.849%
Final val acc for full class for lr = 1e-05 and number of unfreeze layers = 3: 50.959%
Final val acc for full class for lr = 5e-05 and number of unfreeze layers = 3: 86.575%
Final val acc for full class for lr = 0.0001 and number of unfreeze layers = 3: 90.685%
Final val acc for full class for lr = 0.0005 and number of unfreeze layers = 3: 83.288%

Best was: Final val acc for full class for lr = 0.0001 and number of unfreeze layers = 3: 90.685%"""

Final val accuracy for full classification with strategy 1 for different learning rates and number of unfreeze layers
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 205MB/s]


Final val acc for full class for lr = 1e-05 and number of unfreeze layers = 1: 43.836%
Final val acc for full class for lr = 5e-05 and number of unfreeze layers = 1: 85.753%
Final val acc for full class for lr = 0.0001 and number of unfreeze layers = 1: 89.863%
Final val acc for full class for lr = 0.0005 and number of unfreeze layers = 1: 87.397%
Final val acc for full class for lr = 1e-05 and number of unfreeze layers = 2: 43.836%
Final val acc for full class for lr = 5e-05 and number of unfreeze layers = 2: 87.671%
Final val acc for full class for lr = 0.0001 and number of unfreeze layers = 2: 90.137%
Final val acc for full class for lr = 0.0005 and number of unfreeze layers = 2: 86.849%
Final val acc for full class for lr = 1e-05 and number of unfreeze layers = 3: 50.959%
Final val acc for full class for lr = 5e-05 and number of unfreeze layers = 3: 86.575%
Final val acc for full class for lr = 0.0001 and number of unfreeze layers = 3: 90.685%
Final val acc for full class for lr = 

# Strategy 2

In [ ]:
# help funtion
def unfreeze_last_blocks(trainable_blocks, l):

    if l == 0:
        print('Only fc is trainable')
        return

    last_blocks = trainable_blocks[-(l+1):-1]

    for name, block in last_blocks:
        for param in block.parameters():
            param.requires_grad = True

    #print('Unfrozen blocks:', [name for name, _ in last_blocks])

# choose layers to unfreeze

l = 2 # choose amount of unfrozen layers
stages = list(range(1,l+1))

In [ ]:
model = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device) # Only choice (of weights param)?

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

trainable_blocks = []

for name, module in model.named_children():
    has_params = any(p.requires_grad is not None for p in module.parameters())

    if has_params:
        trainable_blocks.append((name, module))

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 37).to(device)
model = model.to(device)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_full:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_full)
    train_acc  = correct / len(train_set_full)


for stage in stages:
    criterion = nn.CrossEntropyLoss()
    unfreeze_last_blocks(trainable_blocks, stage)

    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0

        for images, labels in train_loader_full:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()

        train_loss = running_loss / len(train_set_full)
        train_acc  = correct / len(train_set_full)



model.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader_full:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        # 0-36 depending on breed
        correct += (outputs.argmax(1) == labels).sum().item()

test_acc = correct / len(test_set_full)

In [ ]:
print('Final test accuracy for full classification with two gradually unefreezed layer: ' + str(test_acc*100) + '%') # I get 90.0% doing gradual unfreezing with two layers (~11 min total, takes a lot longer in colab)

### Hyper parameter search for strategy 2

In [ ]:
l_values = [1, 2, 3]
finetune_lr_values = [1e-5, 5e-5, 1e-4, 5e-4]

print(f'Final val accuracy for full classification with strategy 2 for different finetuning learning rates and number of gradually unfrozen layers')
print("Uses the best learning rate from the full classification experiment, i.e. 3e-3, for the fully connected layer's learningrate")
for l in l_values:
    for finetune_lr in finetune_lr_values:
        val_acc = modelGenAndfullClassGradUnfreeze(train_loader_full, l, weight_decay=0, fc_lr=3e-3, finetune_lr=finetune_lr, test_or_val_loader_full = val_loader_full)
        print(f'Final val acc for full class for finetune_lr = {finetune_lr} and number of gradually unfrozen layers = {l}: {val_acc*100:.3f}%')

"""Final val accuracy for full classification with strategy 2 for different finetuning learning rates and number of gradually unfrozen layers
Uses the best learning rate from the full classification experiment, i.e. 3e-3, for the full classification layer's learningrate
Final val acc for full class for finetune_lr = 1e-05 and number of gradually unfrozen layers = 1: 89.315%
Final val acc for full class for finetune_lr = 5e-05 and number of gradually unfrozen layers = 1: 90.137%
Final val acc for full class for finetune_lr = 0.0001 and number of gradually unfrozen layers = 1: 90.411%
Final val acc for full class for finetune_lr = 0.0005 and number of gradually unfrozen layers = 1: 86.027%
Final val acc for full class for finetune_lr = 1e-05 and number of gradually unfrozen layers = 2: 90.685%
Final val acc for full class for finetune_lr = 5e-05 and number of gradually unfrozen layers = 2: 90.959%
Final val acc for full class for finetune_lr = 0.0001 and number of gradually unfrozen layers = 2: 92.055%
Final val acc for full class for finetune_lr = 0.0005 and number of gradually unfrozen layers = 2: 80.822%
Final val acc for full class for finetune_lr = 1e-05 and number of gradually unfrozen layers = 3: 89.863%
Final val acc for full class for finetune_lr = 5e-05 and number of gradually unfrozen layers = 3: 90.959%
Final val acc for full class for finetune_lr = 0.0001 and number of gradually unfrozen layers = 3: 89.041%
Final val acc for full class for finetune_lr = 0.0005 and number of gradually unfrozen layers = 3: 77.808%

Best was: Final val acc for full class for finetune_lr = 0.0001 and number of gradually unfrozen layers = 2: 92.055% (and fc_lr=3e-3)


Final val accuracy for full classification with strategy 2 for different finetuning learning rates and number of gradually unfrozen layers
Uses the best learning rate from the full classification experiment, i.e. 3e-3, for the full classification layer's learningrate
Final val acc for full class for finetune_lr = 1e-05 and number of gradually unfrozen layers = 1: 89.315%
Final val acc for full class for finetune_lr = 5e-05 and number of gradually unfrozen layers = 1: 90.137%
Final val acc for full class for finetune_lr = 0.0001 and number of gradually unfrozen layers = 1: 90.411%
Final val acc for full class for finetune_lr = 0.0005 and number of gradually unfrozen layers = 1: 86.027%
Final val acc for full class for finetune_lr = 1e-05 and number of gradually unfrozen layers = 2: 90.685%
Final val acc for full class for finetune_lr = 5e-05 and number of gradually unfrozen layers = 2: 90.959%
Final val acc for full class for finetune_lr = 0.0001 and number of gradually unfrozen layers 


# Fine-tuning with limited data

## full classification with label fractions

In [ ]:
def stratified_subset(dataset, fraction, seed=42):
    random.seed(seed)
    class_to_indices = defaultdict(list)

    for idx in range(len(dataset)):
        label = dataset[idx][1]
        class_to_indices[label].append(idx)

    selected_indices = []

    for label, indices in class_to_indices.items():
        random.shuffle(indices)
        n_keep = max(1, int(len(indices) * fraction))
        selected_indices.extend(indices[:n_keep])

    random.shuffle(selected_indices)

    return Subset(dataset, selected_indices)

In [ ]:
train_set_100 = stratified_subset(train_set_full, 1.00)
train_set_10  = stratified_subset(train_set_full, 0.10)
train_set_1   = stratified_subset(train_set_full, 0.01)

batch_size = 50
train_loader_100 = DataLoader(train_set_100, batch_size=batch_size, shuffle=True, num_workers=2)
train_loader_10  = DataLoader(train_set_10,  batch_size=batch_size, shuffle=True, num_workers=2)
train_loader_1   = DataLoader(train_set_1,   batch_size=batch_size, shuffle=True, num_workers=2)

print("100% training size:", len(train_set_100))
print("10% training size:", len(train_set_10))
print("1% training size:", len(train_set_1))

100% training size: 3315
10% training size: 329
1% training size: 37


In [ ]:
#Testing full classification with the different data sizes
def modelGeneration():
    model = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device) # Only choice (of weights param)?

    # Matching the tutorial
    for param in model.parameters():
        param.requires_grad = False

    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 37).to(device)
    model.to(device)

    return model

In [ ]:

def fullClassification(model, train_loader, lr, weight_decay = 0): #lr should be 1e-3
    # Training for full classification

    #print(f'Results for fraction: {(len(train_loader.dataset)/3680):.2f}')

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=lr, weight_decay=weight_decay) # Maybe lr should change for greater than 99%

    num_epochs = 2

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()


        train_size = len(train_loader.dataset)

        train_loss = running_loss / train_size
        train_acc  = correct / train_size



        model.eval()
        correct = 0
        with torch.no_grad():
            for images, labels in val_loader_full:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                # 0-36 depending on breed
                correct += (outputs.argmax(1) == labels).sum().item()

        val_acc = correct / len(val_set_full)

        #print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')
    return val_acc

In [ ]:

print(f'Final test accuracy for full classification for different fractions of labelled data')
lr = 1e-3
model = modelGeneration()
test_acc = fullClassification(model, train_loader_1, lr)
print(f'Fraction: {(len(train_loader_1.dataset)/3680):.2f}: {test_acc*100:.3f}%')
model = modelGeneration()
#test_acc = fullClassification(model, train_loader_10, lr)
print(f'Fraction: {(len(train_loader_10.dataset)/3680):.2f}: {test_acc*100:.3f}%')
model = modelGeneration()
#test_acc = fullClassification(model, train_loader_100, lr)
print(f'Fraction: {(len(train_loader_100.dataset)/3680):.2f}: {test_acc*100:.3f}%')




## Fine tune with fractions strategy 1

In [ ]:
def modelGenFullClassStrat1(train_loader, lr ,numUnfreezeLayers, weight_decay=0, test_or_val_loader_full = val_loader_full):
    model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?

    # Matching the tutorial
    for param in model.parameters():
        param.requires_grad = False

    trainable_blocks = []

    for name, module in model.named_children():
        has_params = any(p.requires_grad is not None for p in module.parameters())

        if has_params:
            trainable_blocks.append((name, module))

    l = numUnfreezeLayers

    last_blocks = trainable_blocks[-(l+1):-1]

    for name, block in last_blocks:
        for param in block.parameters():
            param.requires_grad = True

    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 37).to(device)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay) # Maybe lr should change for greater than 99%

    num_epochs = 2

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()

        train_size = len(train_loader.dataset)

        train_loss = running_loss / train_size
        train_acc  = correct / train_size

        model.eval()
        correct = 0
        with torch.no_grad():
            for images, labels in test_or_val_loader_full:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                # 0-36 depending on breed
                correct += (outputs.argmax(1) == labels).sum().item()

        test_or_val_acc = correct / len(test_or_val_loader_full.dataset)

        #print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')
    return test_or_val_acc

## Strategy 2 (gradual unfreezing of layers) with different fractions of labelled data

In [ ]:
def modelGenAndfullClassGradUnfreeze(train_loader, l, weight_decay=0, fc_lr=3e-3, finetune_lr=1e-4, test_or_val_loader_full = val_loader_full):  #OBS, l is not lr
  model = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device) # Only choice (of weights param)?

  # Matching the tutorial
  for param in model.parameters():
      param.requires_grad = False

  trainable_blocks = []

  for name, module in model.named_children():
      has_params = any(p.requires_grad is not None for p in module.parameters())

      if has_params:
          trainable_blocks.append((name, module))

  num_features = model.fc.in_features
  model.fc = nn.Linear(num_features, 37).to(device)
  model = model.to(device)

  stages = list(range(1,l+1))

  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.fc.parameters(), lr=fc_lr, weight_decay=weight_decay) # Maybe lr should change for greater than 99%

  num_epochs = 2

  for epoch in range(num_epochs):
      model.train()
      running_loss = 0.0
      correct = 0

      for images, labels in train_loader:
          images, labels = images.to(device), labels.to(device)

          optimizer.zero_grad()
          outputs = model(images)
          loss = criterion(outputs, labels)
          loss.backward()
          optimizer.step()

          running_loss += loss.item() * images.size(0)
          correct += (outputs.argmax(1) == labels).sum().item()

      train_size = len(train_loader.dataset)
      train_loss = running_loss / train_size
      train_acc  = correct / train_size


  for stage in stages:
      criterion = nn.CrossEntropyLoss()
      unfreeze_last_blocks(trainable_blocks, stage)

      optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=finetune_lr, weight_decay=weight_decay) #reasonable to use different learning rate here

      for epoch in range(num_epochs):
          model.train()
          running_loss = 0.0
          correct = 0

          for images, labels in train_loader:
              images, labels = images.to(device), labels.to(device)

              optimizer.zero_grad()
              outputs = model(images)
              loss = criterion(outputs, labels)
              loss.backward()
              optimizer.step()

              running_loss += loss.item() * images.size(0)
              correct += (outputs.argmax(1) == labels).sum().item()

          train_size = len(train_loader.dataset)
          train_loss = running_loss / train_size
          train_acc  = correct / train_size



  model.eval()
  correct = 0
  with torch.no_grad():
      for images, labels in test_or_val_loader_full:
          images, labels = images.to(device), labels.to(device)
          outputs = model(images)
          # 0-36 depending on breed
          correct += (outputs.argmax(1) == labels).sum().item()

  test_or_val_acc = correct / len(test_or_val_loader_full.dataset)
  return test_or_val_acc

In [ ]:
#returns 2 fraction subsets with the same data split, one with no data aug, and one with data aug
def stratified_subset_noaug_and_aug(dataset_noaug, dataset_aug, fraction, seed=42):
    random.seed(seed)
    class_to_indices = defaultdict(list)

    for idx in range(len(dataset_noaug)):
        label = dataset_noaug[idx][1]
        class_to_indices[label].append(idx)

    selected_indices = []

    for label, indices in class_to_indices.items():
        random.shuffle(indices)
        n_keep = max(1, int(len(indices) * fraction))
        selected_indices.extend(indices[:n_keep])

    random.shuffle(selected_indices)

    return Subset(dataset_noaug, selected_indices), Subset(dataset_aug, selected_indices)

In [ ]:
train_set_100_noaug, train_set_100_aug = stratified_subset_noaug_and_aug(train_set_full,train_set_full_aug, 1.00) #both the aug and noaug set have the same datasplit
train_set_10_noaug, train_set_10_aug  = stratified_subset_noaug_and_aug(train_set_full, train_set_full_aug, 0.10)
train_set_1_noaug, train_set_1_aug   = stratified_subset_noaug_and_aug(train_set_full, train_set_full_aug, 0.01)

batch_size = 50
train_loader_100_noaug = DataLoader(train_set_100_noaug, batch_size=batch_size, shuffle=True, num_workers=2)
train_loader_10_noaug  = DataLoader(train_set_10_noaug,  batch_size=batch_size, shuffle=True, num_workers=2)
train_loader_1_noaug   = DataLoader(train_set_1_noaug,   batch_size=batch_size, shuffle=True, num_workers=2)
train_loader_100_aug = DataLoader(train_set_100_aug, batch_size=batch_size, shuffle=True, num_workers=2)
train_loader_10_aug  = DataLoader(train_set_10_aug,  batch_size=batch_size, shuffle=True, num_workers=2)
train_loader_1_aug   = DataLoader(train_set_1_aug,   batch_size=batch_size, shuffle=True, num_workers=2)

print("noaug sets")
print("100% training size:", len(train_set_100_noaug))
print("10% training size:", len(train_set_10_noaug))
print("1% training size:", len(train_set_1_noaug))

print("aug sets")
print("100% training size:", len(train_set_100_aug))
print("10% training size:", len(train_set_10_aug))
print("1% training size:", len(train_set_1_aug))


noaug sets
100% training size: 3315
10% training size: 329
1% training size: 37
aug sets
100% training size: 3315
10% training size: 329
1% training size: 37


In [ ]:
#l2 regularization test:

weight_decay = 1e-5
print(f'Final test accuracy for full classification for different fractions of labelled data with the best model (strategy 2) with and without {weight_decay} L2 regularization')

#Best model from strategy 2
#2 layers
l = 2 # choose amount of unfrozen layers
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_1_noaug, l, weight_decay=weight_decay)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_1_noaug.dataset)/3680):.2f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_10_noaug, l, weight_decay=weight_decay)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_10_noaug.dataset)/3680):.2f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_100_noaug, l, weight_decay=weight_decay)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_100_noaug.dataset)/3680):.2f}: {test_acc*100:.3f}%')

#For weightdecay 1e-3: 12.29%, 75.55%, 89.40% for fraction 0.01, 0.1, 1.00 respectively
#For weightdecay 1e-4: 16.68%, 75.77%, 89.78% for fraction 0.01, 0.1, 1.00 respectively
#For weightdecay 1e-5: 15.89%, 75.69%, 90.08 for fraction 0.01, 0.1, 1.00 respectively
#For weightdecay 0: 15.59%, 76.59%, 89.26% for fraction 0.01, 0.1, 1.00 respectively


### Final code to produce all results needed for the section

---



In [ ]:
# Final code to produce all results for the fine tuning with limited data part

#Needs fract 1, 0.1 and 0.01 and test in on linear probing and the best strategy 1 and 2 models
print(f'everything uses 2 epochs')
print(f'Final validation accuracy for full classification for different fractions of labelled data for different strategies without data augmentation')


#Linear probing
lr=0.003 #uses best lr from full class experiment
model = modelGeneration()
test_acc = fullClassification(model, train_loader_1_noaug, lr)
print(f'Lin prob: Fraction without dataaug: {(len(train_loader_1_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')

model = modelGeneration()
test_acc = fullClassification(model, train_loader_10_noaug, lr)
print(f'Lin prob: Fraction without dataaug: {(len(train_loader_10_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')

model = modelGeneration()
test_acc = fullClassification(model, train_loader_100_noaug, lr)
print(f'Lin Prob: Fraction without dataaug: {(len(train_loader_100_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')

#Best model from strategy 1
#Best was: Final val acc for full class for lr = 0.0001 and number of unfreeze layers = 3: 90.685%"""
lr=0.0001
numUnfreezeLayers = 3
test_acc = modelGenFullClassStrat1(train_loader_1_noaug, lr ,numUnfreezeLayers)
print(f'Strategy 1: Fraction without dataaug: {(len(train_loader_1_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')
test_acc = modelGenFullClassStrat1(train_loader_10_noaug, lr ,numUnfreezeLayers)
print(f'Strategy 1: Fraction without dataaug: {(len(train_loader_10_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')
test_acc = modelGenFullClassStrat1(train_loader_100_noaug, lr ,numUnfreezeLayers)
print(f'Strategy 1: Fraction without dataaug: {(len(train_loader_100_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')


#Best model from strategy 2
#Best was: Final val acc for full class for finetune_lr = 0.0001 and number of gradually unfrozen layers = 2: 92.055% (and fc_lr=3e-3)

l = 2 # choose amount of unfrozen layers
fc_lr=0.003
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_1_noaug, l,fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction without dataaug: {(len(train_loader_1_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_10_noaug, l,fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction without dataaug: {(len(train_loader_10_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_100_noaug, l,fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction without dataaug: {(len(train_loader_100_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')

print(f'Final validation accuracy for full classification for different fractions of labelled data with the best model (strategy 2) with and without data augmentation')

#Best model from strategy 2

l = 2 # choose amount of unfrozen layers
fc_lr=0.003
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_1_aug, l,fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with dataaug: {(len(train_loader_1_aug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_10_aug, l,fc_lr=fc_lr,finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with dataaug: {(len(train_loader_10_aug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_100_aug, l,fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with dataaug: {(len(train_loader_100_aug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')

#-------------------------------weight decay bit --------------------------------

weight_decay = 1e-2
print(f'Final validation accuracy for full classification for different fractions of labelled data with the best model (strategy 2) with and without {weight_decay} L2 regularization')

#Best model from strategy 2
#Best was: Final val acc for full class for finetune_lr = 0.0001 and number of gradually unfrozen layers = 2: 92.055% (and fc_lr=3e-3)
l = 2 # choose amount of unfrozen layers
fc_lr=0.003
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_1_noaug, l,weight_decay=weight_decay, fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_1_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_10_noaug, l,weight_decay=weight_decay,fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_10_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_100_noaug, l,weight_decay=weight_decay,fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_100_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')


weight_decay = 1e-3
print(f'Final validation accuracy for full classification for different fractions of labelled data with the best model (strategy 2) with and without {weight_decay} L2 regularization')

#Best model from strategy 2
#Best was: Final val acc for full class for finetune_lr = 0.0001 and number of gradually unfrozen layers = 2: 92.055% (and fc_lr=3e-3)
l = 2 # choose amount of unfrozen layers
fc_lr=0.003
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_1_noaug, l,weight_decay=weight_decay, fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_1_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_10_noaug, l,weight_decay=weight_decay,fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_10_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_100_noaug, l,weight_decay=weight_decay,fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_100_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')

weight_decay = 1e-4
print(f'Final validation accuracy for full classification for different fractions of labelled data with the best model (strategy 2) with and without {weight_decay} L2 regularization')

#Best model from strategy 2
#Best was: Final val acc for full class for finetune_lr = 0.0001 and number of gradually unfrozen layers = 2: 92.055% (and fc_lr=3e-3)
l = 2 # choose amount of unfrozen layers
fc_lr=0.003
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_1_noaug, l,weight_decay=weight_decay, fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_1_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_10_noaug, l,weight_decay=weight_decay,fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_10_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_100_noaug, l,weight_decay=weight_decay,fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_100_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')

weight_decay = 1e-5
print(f'Final validation accuracy for full classification for different fractions of labelled data with the best model (strategy 2) with and without {weight_decay} L2 regularization')

#Best model from strategy 2
#Best was: Final val acc for full class for finetune_lr = 0.0001 and number of gradually unfrozen layers = 2: 92.055% (and fc_lr=3e-3)
l = 2 # choose amount of unfrozen layers
fc_lr=0.003
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_1_noaug, l,weight_decay=weight_decay, fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_1_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_10_noaug, l,weight_decay=weight_decay,fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_10_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_100_noaug, l,weight_decay=weight_decay,fc_lr=fc_lr, finetune_lr = 0.0001)
print(f'Strategy 2: Fraction with L2 reg: {(len(train_loader_100_noaug.dataset)/len(train_loader_full.dataset)):.3f}: {test_acc*100:.3f}%')


#Results for those without L2 regularization can be found in the earlier section

#Combination of best model (strategy 2) with both data augmentation and
#print(f'Final test accuracy for full classification with diff fracs of data and Combination of best model (strategy 2) with both data augmentation and 1e-4 L2 regularization')

#Best model from strategy 2
#2 layers
# weight_decay = 1e-4
# l = 2 # choose number of unfrozen layers
# test_acc = modelGenAndfullClassGradUnfreeze(train_loader_100_aug, l, weight_decay=weight_decay)
# print(f'Final combo model 100%data: {(len(train_loader_100_aug.dataset)/3680):.2f}: {test_acc*100:.3f}%')

# test_acc = modelGenAndfullClassGradUnfreeze(train_loader_10_aug, l, weight_decay=weight_decay)
# print(f'Final combo model 10%data: {(len(train_loader_10_aug.dataset)/3680):.2f}: {test_acc*100:.3f}%')

# test_acc = modelGenAndfullClassGradUnfreeze(train_loader_1_aug, l, weight_decay=weight_decay)
# print(f'Final combo model 1%data: {(len(train_loader_1_aug.dataset)/3680):.2f}: {test_acc*100:.3f}%')

#Results
"""everything uses 2 epochs
Final validation accuracy for full classification for different fractions of labelled data for different strategies without data augmentation
Lin prob: Fraction without dataaug: 0.011: 17.260%
Lin prob: Fraction without dataaug: 0.099: 66.575%
Lin Prob: Fraction without dataaug: 1.000: 89.315%
Strategy 1: Fraction without dataaug: 0.011: 3.014%
Strategy 1: Fraction without dataaug: 0.099: 43.836%
Strategy 1: Fraction without dataaug: 1.000: 90.411%
Strategy 2: Fraction without dataaug: 0.011: 35.068%
Strategy 2: Fraction without dataaug: 0.099: 81.096%
Strategy 2: Fraction without dataaug: 1.000: 90.411%
Final validation accuracy for full classification for different fractions of labelled data with the best model (strategy 2) with and without data augmentation
Strategy 2: Fraction with dataaug: 0.011: 40.000%
Strategy 2: Fraction with dataaug: 0.099: 79.178%
Strategy 2: Fraction with dataaug: 1.000: 90.685%
Final validation accuracy for full classification for different fractions of labelled data with the best model (strategy 2) with and without 0.01 L2 regularization
Strategy 2: Fraction with L2 reg: 0.011: 31.507%
Strategy 2: Fraction with L2 reg: 0.099: 83.836%
Strategy 2: Fraction with L2 reg: 1.000: 91.233%
Final validation accuracy for full classification for different fractions of labelled data with the best model (strategy 2) with and without 0.001 L2 regularization
Strategy 2: Fraction with L2 reg: 0.011: 34.795%
Strategy 2: Fraction with L2 reg: 0.099: 83.288%
Strategy 2: Fraction with L2 reg: 1.000: 90.411%
Final validation accuracy for full classification for different fractions of labelled data with the best model (strategy 2) with and without 0.0001 L2 regularization
Strategy 2: Fraction with L2 reg: 0.011: 34.521%
Strategy 2: Fraction with L2 reg: 0.099: 81.644%
Strategy 2: Fraction with L2 reg: 1.000: 90.959%
Final validation accuracy for full classification for different fractions of labelled data with the best model (strategy 2) with and without 1e-05 L2 regularization
Strategy 2: Fraction with L2 reg: 0.011: 33.425%
Strategy 2: Fraction with L2 reg: 0.099: 83.014%
Strategy 2: Fraction with L2 reg: 1.000: 89.589%"""

everything uses 2 epochs
Final validation accuracy for full classification for different fractions of labelled data for different strategies without data augmentation
Lin prob: Fraction without dataaug: 0.011: 17.260%
Lin prob: Fraction without dataaug: 0.099: 66.575%
Lin Prob: Fraction without dataaug: 1.000: 89.315%
Strategy 1: Fraction without dataaug: 0.011: 3.014%
Strategy 1: Fraction without dataaug: 0.099: 43.836%
Strategy 1: Fraction without dataaug: 1.000: 90.411%
Strategy 2: Fraction without dataaug: 0.011: 35.068%
Strategy 2: Fraction without dataaug: 0.099: 81.096%
Strategy 2: Fraction without dataaug: 1.000: 90.411%
Final validation accuracy for full classification for different fractions of labelled data with the best model (strategy 2) with and without data augmentation
Strategy 2: Fraction with dataaug: 0.011: 40.000%
Strategy 2: Fraction with dataaug: 0.099: 79.178%
Strategy 2: Fraction with dataaug: 1.000: 90.685%
Final validation accuracy for full classification for 

'OBS OLD RUN BELOW:\neverything uses 2 epochs\nFinal test accuracy for full classification for different fractions of labelled data for different strategies without data augmentation\nLin prob: Fraction without dataaug: 0.01: 4.197%\nLin prob: Fraction without dataaug: 0.10: 38.021%\nLin Prob: Fraction without dataaug: 1.00: 84.137%\nStrategy 1: Fraction without dataaug: 0.01: 3.816%\nStrategy 1: Fraction without dataaug: 0.10: 47.915%\nStrategy 1: Fraction without dataaug: 1.00: 87.953%\nStrategy 2: Fraction without dataaug: 0.01: 16.380%\nStrategy 2: Fraction without dataaug: 0.10: 77.324%\nStrategy 2: Fraction without dataaug: 1.00: 90.106%\nFinal test accuracy for full classification for different fractions of labelled data with the best model (strategy 2) with and without data augmentation\nStrategy 2: Fraction with dataaug: 0.01: 13.873%\nStrategy 2: Fraction with dataaug: 0.10: 76.424%\nStrategy 2: Fraction with dataaug: 1.00: 88.117%\nFinal test accuracy for full classification



# Fine-tuning with imbalanced classes

##Create imbalanced data and train the model on it

In [ ]:
cats = [] # contains the labels that has cats
for i, breed in enumerate(train_set_full.classes):
  if breed[0].isupper():
    cats.append(i) # This works since all cat breeds start with an uppercase letter in the Oxford-IIIT-pet dataset, whereas dogs start with lowercase

indices_for_kept_images = []
percentage_kept = 0.20

for index in range(len(train_set_full)):
  label = train_set_full[index][1]
  if label in cats:
    if random.random() < percentage_kept:
      indices_for_kept_images.append(index)
  else:
    indices_for_kept_images.append(index)

train_set_imbalanced = Subset(train_set_full, indices_for_kept_images)
train_loader_imbalanced = DataLoader(train_set_imbalanced, batch_size=50, shuffle=True,  num_workers=2) # Choosing num_workers=2 when running in Colab

# Training the unbalanced model
model_imbalanced = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device)

# Matching the tutorial
for param in model_imbalanced.parameters():
    param.requires_grad = False

# Get the number of input features for the original fc layer
num_features = model_imbalanced.fc.in_features
# Replace the fc layer with a new one for the number of classes in the dataset
model_imbalanced.fc = nn.Linear(num_features, len(train_set_full.classes)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_imbalanced.fc.parameters(), lr=0.001)
n_epochs = 2

for epoch in range(n_epochs):
  model_imbalanced.train()
  for images, labels in train_loader_imbalanced:
    images = images.to(device)
    labels = labels.to(device)
    optimizer.zero_grad()
    outputs = model_imbalanced(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

AttributeError: 'Subset' object has no attribute 'classes'

## Evaluating F1 scores based on test data

In [ ]:
model_imbalanced.eval()

# Creating lists for storing true and guessed breeds
y_true = []
y_pred = []

with torch.no_grad():
  for images, labels in test_loader_full:
    images = images.to(device)
    labels = labels.to(device)
    outputs = model_imbalanced(images)
    y_true.extend(labels.cpu().tolist())
    y_pred.extend(outputs.argmax(1).cpu().tolist())

print(classification_report(y_true,y_pred,target_names=train_set_full.classes))


## Compensating with weighted cross entropy

In [ ]:
from torch.jit import optimize_for_inference
weights = torch.ones(37)

for breed in cats:
  weights[breed] = 5.0
weights = weights.to(device)

# Creating uppdated loss funciton
criterion_weighted = nn.CrossEntropyLoss(weight=weights)

# Weighted cross entropy model
model_wce = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device)

for param in model_wce.parameters():
  param.requires_grad = False

n_features = model_wce.fc.in_features
model_wce.fc = nn.Linear(n_features, len(train_set_full.classes)).to(device)

optimizer_wce = torch.optim.Adam(model_wce.fc.parameters(), lr=0.001)

for epoch in range(n_epochs):
  model_wce.train()
  for images, labels in train_loader_imbalanced:
    images = images.to(device)
    labels = labels.to(device)
    optimizer_wce.zero_grad()
    outputs = model_wce(images)
    loss = criterion_weighted(outputs, labels)
    loss.backward()
    optimizer_wce.step()

model_wce.eval()
y_true_wce = []
y_pred_wce = []

with torch.no_grad():
  for images, labels in test_loader_full:
    images = images.to(device)
    labels = labels.to(device)
    outputs = model_wce(images)
    y_true_wce.extend(labels.cpu().tolist())
    y_pred_wce.extend(outputs.argmax(1).cpu().tolist())

print(classification_report(y_true_wce,y_pred_wce,target_names=train_set_full.classes))


## Performing accuracy test

In [ ]:
#Best model from strategy 2
#2 layers
l = 2 # choose amount of unfrozen layers
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_imbalanced, l)
print(f'Strategy 2: Fraction without dataaug: {(len(train_loader_imbalanced.dataset)/3680):.2f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_imbalanced, l)
print(f'Strategy 2: Fraction without dataaug: {(len(train_loader_imbalanced.dataset)/3680):.2f}: {test_acc*100:.3f}%')
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_imbalanced, l)
print(f'Strategy 2: Fraction without dataaug: {(len(train_loader_imbalanced.dataset)/3680):.2f}: {test_acc*100:.3f}%')

Strategy 2: Fraction without dataaug: 0.19: 84.383%
Strategy 2: Fraction without dataaug: 0.19: 83.401%
Strategy 2: Fraction without dataaug: 0.19: 83.483%


#  Semi-supervised learning

In [ ]:

#Returns split of training data with fraction labelled data and (1 - fraction) unlabelled data
def stratified_labelled_unlabelled_split(dataset, fraction, seed=42):
    random.seed(seed)
    class_to_indices = defaultdict(list)

    for idx in range(len(dataset)):
        label = dataset[idx][1]
        class_to_indices[label].append(idx)

    labelled_data_indices = []
    unlabelled_data_indices = []

    for label, indices in class_to_indices.items():
        random.shuffle(indices)
        n_keep = max(1, int(len(indices) * fraction))
        labelled_data_indices.extend(indices[:n_keep]) #adds first n_keep indices
        unlabelled_data_indices.extend(indices[n_keep:]) #adds all indices after first n_keep indices => different indices (images) than the labelled data


    random.shuffle(labelled_data_indices)
    random.shuffle(unlabelled_data_indices)

    return Subset(dataset, labelled_data_indices), Subset(dataset, unlabelled_data_indices)

In [ ]:
def update_teacher(student, teacher, alpha=0.99): # Can start with lower alpha like 0.99 and increase during training (or towards end)
    with torch.no_grad():
        for t_param, s_param in zip(teacher.parameters(), student.parameters()):
            t_param.data = (alpha * t_param.data) + ((1 - alpha) * s_param.data)


In [ ]:
def get_augment_transform(): # For consist. reg. training (mean teacher)
    return transforms.Compose([
        transforms.RandomResizedCrop(224),
        #transforms.RandomRotation(10),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])


In [ ]:
def alpha_schedule(epoch, T1=1, T2=4, alpha_f=0.5):
    if epoch < T1:
        alpha = 0
        return alpha

    elif epoch < T2:
        alpha = (epoch - T1) / (T2 - T1) * alpha_f
        return alpha

    else:
        alpha = alpha_f
        return alpha

In [ ]:

def train_mean_teacher(student, teacher, labeled_loader, unlabeled_loader, num_epochs=10, lr=1e-3, ema_alpha=0.99, curve=None):
    optimizer = torch.optim.Adam(student.fc.parameters(), lr=lr)
    supervised_criterion = nn.CrossEntropyLoss()
    consistency_criterion = nn.MSELoss() # MSE outperforms KL-div and others according to paper

    for epoch in range(num_epochs):
        student.train()
        teacher.train()

        total_loss = 0.0
        correct = 0
        n_labeled = 0
        i = 0

        labeled_iter = iter(labeled_loader)

        for aug1, aug2 in unlabeled_loader:
            try:
                images, labels = next(labeled_iter)
            except StopIteration:
                labeled_iter = iter(labeled_loader)
                images, labels = next(labeled_iter)

            images, labels = images.to(device), labels.to(device)
            aug1, aug2 = aug1.to(device), aug2.to(device)


            student_out = student(images)
            supervised_loss = supervised_criterion(student_out, labels)

            student_unlabeled = torch.softmax(student(aug1), dim=1)
            with torch.no_grad():
                teacher_unlabeled = torch.softmax(teacher(aug2), dim=1)

            consistency_loss = consistency_criterion(student_unlabeled, teacher_unlabeled)

            # Combined loss
            loss = supervised_loss + alpha_schedule(epoch) * consistency_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            update_teacher(student, teacher, alpha=ema_alpha)

            total_loss += loss.item() * images.size(0)
            correct += (student_out.argmax(1) == labels).sum().item()
            n_labeled += images.size(0)


        train_acc = correct / n_labeled


        teacher.eval()
        correct = 0
        with torch.no_grad():
            for images, labels in test_loader_full:
                images, labels = images.to(device), labels.to(device)
                correct += (teacher(images).argmax(1) == labels).sum().item()

        test_acc = correct / len(test_set_full)
        print(f'Epoch [{epoch+1:02d}/{num_epochs}]' f'Loss: {total_loss/n_labeled:.4f}' f'Train Acc: {train_acc:.4f}' f'Test Acc (teacher): {test_acc:.4f}')

        if curve is not None:
            curve.append(test_acc)

    return student, teacher

In [ ]:
class TwoAugDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        return self.transform(img), self.transform(img)

In [ ]:
labeled_subset_mt, unlabeled_subset_mt = stratified_labelled_unlabelled_split(train_set_full, fraction=0.1)

labeled_loader_mt = DataLoader(labeled_subset_mt, batch_size=32, shuffle=True, num_workers=0)

train_set_raw = OxfordIIITPet(root='data', split='trainval', target_types='category', transform=None, download=False)
unlabeled_indices_mt = unlabeled_subset_mt.indices
unlabeled_set_mt = TwoAugDataset(torch.utils.data.Subset(train_set_raw, unlabeled_indices_mt), get_augment_transform())
unlabeled_loader_mt = DataLoader(unlabeled_set_mt, batch_size=32, shuffle=True, num_workers=0)

student = torchvision.models.resnet18(weights='IMAGENET1K_V1')
for param in student.parameters():
    param.requires_grad = False
student.fc = nn.Linear(student.fc.in_features, 37)
student = student.to(device)

import copy
teacher = copy.deepcopy(student)
for param in teacher.parameters():
    param.requires_grad = False

student, teacher = train_mean_teacher(student, teacher, labeled_loader_mt, unlabeled_loader_mt, num_epochs=3)

In [ ]:
def train_pseudo_label_epoch(model, label_loader, unlabel_loader, optimizer, criterion, epoch):
    model.train()
    alpha = alpha_schedule(epoch)
    total_loss = 0

    labeled_iter = iter(label_loader)

    for x_unlab, _ in unlabel_loader:
        try:
            x_lab, y_lab = next(labeled_iter)
        except StopIteration:
            labeled_iter = iter(label_loader)
            x_lab, y_lab = next(labeled_iter)

        x_lab, y_lab, x_unlab = x_lab.to(device), y_lab.to(device), x_unlab.to(device)

        optimizer.zero_grad()

        # normal labelled data
        outputs_lab = model(x_lab)
        loss_lab = criterion(outputs_lab, y_lab)

        # Unlabelled data

        with torch.no_grad():
            outputs_unlab = model(x_unlab)
            pseudo_labels = torch.argmax(outputs_unlab, dim = 1)

        # train with pseudo labels
        outputs_unlab2 = model(x_unlab)
        loss_unlab = criterion(outputs_unlab2, pseudo_labels)

        loss = loss_lab + alpha * loss_unlab

        loss.backward()
        optimizer.step()

        total_loss += loss.item()


    return total_loss / len(label_loader)


In [ ]:
def train_pseudo_label_epoch(model, label_loader, unlabel_loader, optimizer, criterion, epoch):
    model.train()
    alpha = alpha_schedule(epoch)
    total_loss = 0

    labeled_iter = iter(label_loader)

    for x_unlab, _ in unlabel_loader:
        try:
            x_lab, y_lab = next(labeled_iter)
        except StopIteration:
            labeled_iter = iter(label_loader)
            x_lab, y_lab = next(labeled_iter)

        x_lab, y_lab, x_unlab = x_lab.to(device), y_lab.to(device), x_unlab.to(device)

        optimizer.zero_grad()

        # normal labelled data
        outputs_lab = model(x_lab)
        loss_lab = criterion(outputs_lab, y_lab)

        # Unlabelled data

        with torch.no_grad():
            outputs_unlab = model(x_unlab)
            pseudo_labels = torch.argmax(outputs_unlab, dim = 1)

        # train with pseudo labels
        outputs_unlab2 = model(x_unlab)
        loss_unlab = criterion(outputs_unlab2, pseudo_labels)

        loss = loss_lab + alpha * loss_unlab

        loss.backward()
        optimizer.step()

        total_loss += loss.item()


    return total_loss / len(label_loader)


In [ ]:
def compute_accuracy(model, data_loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in data_loader:
            x = x.to(device)
            y = y.to(device)

            outputs = model(x)

            predictions = torch.argmax(outputs, dim=1)

            correct += (predictions == y).sum().item()
            total += y.size(0)

    accuracy = correct / total

    return accuracy

In [ ]:
def pseudo_label_training(model, label_loader, unlabel_loader, epochs, lr, curve=None):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.fc.parameters(), lr) # Maybe lr should change for greater than 99%

    for epoch in range(epochs):
        loss = train_pseudo_label_epoch(model, label_loader, unlabel_loader, optimizer, criterion, epoch)

        test_acc = compute_accuracy(model, test_loader_full)
        print(f"Epoch {epoch+1}/{epochs}, " f"alpha={alpha_schedule(epoch):.3f}, " f"loss={loss:.4f}," f"test_acc={test_acc*100:.2f}%")

        if curve is not None:
            curve.append(test_acc)


    return model


In [ ]:
labeled_subset_pl, unlabeled_subset_pl = stratified_labelled_unlabelled_split(train_set_full, fraction=0.1)

label_loader_pl   = DataLoader(labeled_subset_pl,   batch_size=32, shuffle=True,  num_workers=0)
unlabel_loader_pl = DataLoader(unlabeled_subset_pl,  batch_size=32, shuffle=True,  num_workers=0)

model_pl = torchvision.models.resnet18(weights='IMAGENET1K_V1')
for param in model_pl.parameters():
    param.requires_grad = False
model_pl.fc = nn.Linear(model_pl.fc.in_features, 37)
model_pl = model_pl.to(device)

model_pl = pseudo_label_training(model_pl, label_loader_pl, unlabel_loader_pl, epochs=3, lr=1e-3)

In [ ]:
# Full data baseline
model_full = torchvision.models.resnet18(weights='IMAGENET1K_V1')

for param in model_full.parameters():
    param.requires_grad = False

model_full.fc = nn.Linear(model_full.fc.in_features, 37)
model_full = model_full.to(device)
optimizer_full = torch.optim.Adam(model_full.fc.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    model_full.train()
    for images, labels in train_loader_full:
        images, labels = images.to(device), labels.to(device)
        optimizer_full.zero_grad()
        loss = criterion(model_full(images), labels)
        loss.backward()
        optimizer_full.step()
full_data_acc = compute_accuracy(model_full, test_loader_full)
print(f'Full data baseline accuracy: {full_data_acc:.4f}')

# Experiment 1

In [ ]:
fractions = [0.8, 0.6, 0.4, 0.2, 0.1, 0.05, 0.03, 0.01]
results_mt = []
results_pl = []
results_sup = []

for fraction in fractions:
    print(f'\n Fraction: {fraction}')

    labeled_subset, unlabeled_subset = stratified_labelled_unlabelled_split(train_set_full, fraction=fraction)
    label_loader   = DataLoader(labeled_subset,   batch_size=32, shuffle=True,  num_workers=0)
    unlabel_loader = DataLoader(unlabeled_subset, batch_size=32, shuffle=True,  num_workers=0)

    # Mean Teacher
    print('Mean Teacher:')
    unlabeled_set_mt = TwoAugDataset(torch.utils.data.Subset(train_set_raw, unlabeled_subset.indices), get_augment_transform())
    unlabeled_loader_mt = DataLoader(unlabeled_set_mt, batch_size=32, shuffle=True, num_workers=0)

    student = torchvision.models.resnet18(weights='IMAGENET1K_V1')
    for param in student.parameters():
        param.requires_grad = False
    student.fc = nn.Linear(student.fc.in_features, 37)
    student = student.to(device)
    teacher = copy.deepcopy(student)
    for param in teacher.parameters():
        param.requires_grad = False

    student, teacher = train_mean_teacher(student, teacher, label_loader, unlabeled_loader_mt, num_epochs=10)
    results_mt.append(compute_accuracy(teacher, test_loader_full))

    # Pseudo Labeling
    print('Pseudo Labeling:')
    model_pl = torchvision.models.resnet18(weights='IMAGENET1K_V1')
    for param in model_pl.parameters():
        param.requires_grad = False
    model_pl.fc = nn.Linear(model_pl.fc.in_features, 37)
    model_pl = model_pl.to(device)

    model_pl = pseudo_label_training(model_pl, label_loader, unlabel_loader, epochs=10, lr=1e-3)
    results_pl.append(compute_accuracy(model_pl, test_loader_full))

    # Labeled-only baseline
    model_sup = torchvision.models.resnet18(weights='IMAGENET1K_V1')
    for param in model_sup.parameters():
        param.requires_grad = False
    model_sup.fc = nn.Linear(model_sup.fc.in_features, 37)
    model_sup = model_sup.to(device)
    optimizer_sup = torch.optim.Adam(model_sup.fc.parameters(), lr=1e-3)
    for epoch in range(10):
        model_sup.train()
        for images, labels in label_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer_sup.zero_grad()
            loss = criterion(model_sup(images), labels)
            loss.backward()
            optimizer_sup.step()
    results_sup.append(compute_accuracy(model_sup, test_loader_full))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot([f * 100 for f in fractions], results_mt, marker='o', label='Mean Teacher')
plt.plot([f * 100 for f in fractions], results_pl, marker='o', label='Pseudo Labeling')
plt.plot([f * 100 for f in fractions], results_sup, marker='o', label='Labeled only')
plt.axhline(y=full_data_acc, linestyle='--', label='Full data baseline')
plt.xlabel('Labeled data (%)')
plt.ylabel('Test Accuracy')
plt.title('Semi-Supervised Learning: Mean Teacher vs Pseudo Labeling')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Experiment 2

In [ ]:
fraction = 0.05
num_epochs = 50

labeled_subset, unlabeled_subset = stratified_labelled_unlabelled_split(train_set_full, fraction=fraction)
label_loader   = DataLoader(labeled_subset,   batch_size=32, shuffle=True,  num_workers=0)
unlabel_loader = DataLoader(unlabeled_subset, batch_size=32, shuffle=True,  num_workers=0)

curve_mt = []
curve_pl = []
curve_sup = []

# Mean Teacher
print('Training Mean Teacher')
unlabeled_set_mt = TwoAugDataset(torch.utils.data.Subset(train_set_raw, unlabeled_subset.indices), get_augment_transform())
unlabeled_loader_mt = DataLoader(unlabeled_set_mt, batch_size=32, shuffle=True, num_workers=0)

student = torchvision.models.resnet18(weights='IMAGENET1K_V1')
for param in student.parameters():
    param.requires_grad = False
student.fc = nn.Linear(student.fc.in_features, 37)
student = student.to(device)
teacher = copy.deepcopy(student)
for param in teacher.parameters():
    param.requires_grad = False

student, teacher = train_mean_teacher(student, teacher, label_loader, unlabeled_loader_mt, num_epochs=num_epochs, curve=curve_mt)

# Pseudo-Labeling
print('Training Pseudo-Labeling')
model_pl = torchvision.models.resnet18(weights='IMAGENET1K_V1')
for param in model_pl.parameters():
    param.requires_grad = False
model_pl.fc = nn.Linear(model_pl.fc.in_features, 37)
model_pl = model_pl.to(device)

model_pl = pseudo_label_training(model_pl, label_loader, unlabel_loader, epochs=num_epochs, lr=1e-3, curve=curve_pl)


# Only labeled
model_sup = torchvision.models.resnet18(weights='IMAGENET1K_V1')
for param in model_sup.parameters():
    param.requires_grad = False
model_sup.fc = nn.Linear(model_sup.fc.in_features, 37)
model_sup = model_sup.to(device)
optimizer_sup = torch.optim.Adam(model_sup.fc.parameters(), lr=1e-3)
for epoch in range(num_epochs):
    model_sup.train()
    for images, labels in label_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_sup.zero_grad()
        loss = criterion(model_sup(images), labels)
        loss.backward()
        optimizer_sup.step()
    curve_sup.append(compute_accuracy(model_sup, test_loader_full))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), curve_mt, label='Mean Teacher')
plt.plot(range(1, num_epochs + 1), curve_pl, label='Pseudo Labeling')
plt.plot(range(1, num_epochs + 1), curve_sup, label='Labeled only')
plt.axhline(y=full_data_acc, linestyle='--', label='Full data baseline')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.title(f'Mean Teacher vs Pseudo Labeling ({int(fraction*100)}% labeled data)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Strategy 2 applied to Semi-Supervised Learning

In [ ]:
def create_staged_model(num_classes=37):
    model = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device)

    for param in model.parameters():
        param.requires_grad = False

    trainable_blocks = []
    for name, module in model.named_children():
        if list(module.parameters()):
            trainable_blocks.append((name, module))

    model.fc = nn.Linear(model.fc.in_features, num_classes).to(device)
    return model, trainable_blocks

def unfreeze_last_blocks(trainable_blocks, l):

    if l == 0:
        return

    last_blocks = trainable_blocks[-(l+1):-1]
    for name, block in last_blocks:
        for param in block.parameters():
            param.requires_grad = True


In [ ]:
def train_ssl_unified(method, student, student_blocks, labeled_loader, unlabeled_loader, test_loader, teacher=None, teacher_blocks=None, num_stages=3, epochs_per_stage=15):
    supervised_criterion = nn.CrossEntropyLoss()
    consistency_criterion = nn.MSELoss()
    test_accuracies = []

    stages = list(range(0, num_stages))
    global_epoch = 0

    for stage in stages:
        print(f"\n STAGE {stage}: {method.upper()}")

        unfreeze_last_blocks(student_blocks, stage)
        if teacher:
            unfreeze_last_blocks(teacher_blocks, stage)

        # Optimizer Setup
        backbone_params = [p for n, p in student.named_parameters() if p.requires_grad and not n.startswith('fc')]
        fc_params = student.fc.parameters()

        if len(backbone_params) > 0:
            optimizer = torch.optim.Adam([{'params': backbone_params, 'lr': 1e-4}, {'params': fc_params, 'lr': 1e-3}])
        else:
            optimizer = torch.optim.Adam(fc_params, lr=1e-3)

        for epoch in range(epochs_per_stage):
            student.train()
            if teacher:
                teacher.train()

            current_cons_weight = alpha_schedule(global_epoch)

            total_loss = 0.0
            correct = 0
            n_labeled = 0

            # Correct loading
            if method == 'baseline':
                main_loader = labeled_loader
            else:
                main_loader = unlabeled_loader
                labeled_iter = iter(labeled_loader)

            for batch_data in main_loader:

                # Get Labeled Data
                if method == 'baseline':
                    images, labels = batch_data
                else:
                    # go through the labeled dataset repeatedly
                    try:
                        images, labels = next(labeled_iter)
                    except StopIteration:
                        labeled_iter = iter(labeled_loader)
                        images, labels = next(labeled_iter)

                    # Get Unlabeled Data
                    if method == 'mean_teacher':
                        aug1, aug2 = batch_data
                        aug1, aug2 = aug1.to(device), aug2.to(device)
                    elif method == 'pseudo_label':
                        aug1, _ = batch_data
                        aug1 = aug1.to(device)

                images, labels = images.to(device), labels.to(device)

                # Supervised Pass
                student_out = student(images)
                loss_lab = supervised_criterion(student_out, labels)
                loss = loss_lab

                # Semi-Supervised Pass
                if method == 'pseudo_label':
                    with torch.no_grad():
                        outputs_unlab = student(aug1)
                        pseudo_labels = torch.argmax(outputs_unlab, dim=1)

                    outputs_unlab2 = student(aug1)
                    loss_unlab = supervised_criterion(outputs_unlab2, pseudo_labels)
                    loss = loss_lab + current_cons_weight * loss_unlab

                elif method == 'mean_teacher':
                    student_unlabeled = torch.softmax(student(aug1), dim=1)
                    with torch.no_grad():
                        teacher_unlabeled = torch.softmax(teacher(aug2), dim=1)

                    consistency_loss = consistency_criterion(student_unlabeled, teacher_unlabeled)
                    loss = loss_lab + current_cons_weight * consistency_loss

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                # Teacher EMA
                if method == 'mean_teacher':
                    update_teacher(student, teacher, alpha=0.99)

                total_loss += loss.item() * images.size(0)
                correct += (student_out.argmax(1) == labels).sum().item()
                n_labeled += images.size(0)

            eval_model = teacher if method == 'mean_teacher' else student
            eval_model.eval()
            test_correct = 0
            with torch.no_grad():
                for test_imgs, test_lbls in test_loader:
                    test_imgs, test_lbls = test_imgs.to(device), test_lbls.to(device)
                    test_correct += (eval_model(test_imgs).argmax(1) == test_lbls).sum().item()

            test_acc = test_correct / len(test_loader.dataset)
            test_accuracies.append(test_acc)


            print(f"Global Epoch [{global_epoch+1:02d}] (Stage {stage}) "f"Weight: {current_cons_weight:.3f} " f"Train Acc: {correct/n_labeled:.4f}, Test Acc: {test_acc:.4f}")
            global_epoch += 1

    return test_accuracies

In [ ]:

fraction = 0.05

labeled_subset, unlabeled_subset = stratified_labelled_unlabelled_split(train_set_full, fraction=fraction)

labeled_loader_mt = DataLoader(labeled_subset, batch_size=32, shuffle=True, num_workers=0)

# Needs raw dataset to prevent double transform
train_set_raw = torchvision.datasets.OxfordIIITPet(root='data', split='trainval', target_types='category', transform=None, download=False)

unlabeled_set_mt = TwoAugDataset(
    torch.utils.data.Subset(train_set_raw, unlabeled_subset.indices),
    get_augment_transform()
)
unlabeled_loader_mt = DataLoader(unlabeled_set_mt, batch_size=32, shuffle=True, num_workers=0)



model_baseline, blocks_baseline = create_staged_model()

model_pl, blocks_pl = create_staged_model()

student_mt, blocks_mt = create_staged_model()
teacher_mt = copy.deepcopy(student_mt)

# extract blocks from teacher so it can be unfrozen in sync
teacher_blocks = []
for name, module in teacher_mt.named_children():
    if list(module.parameters()):
        teacher_blocks.append((name, module))


num_stages = 3
epochs_per_stage = 17
total_epochs = num_stages * epochs_per_stage

unlabeled_loader_plain = DataLoader(unlabeled_subset, batch_size=32, shuffle=True, num_workers=0)

baseline_accs = train_ssl_unified('baseline', model_baseline, blocks_baseline, labeled_loader_mt, unlabeled_loader_mt, test_loader_full, num_stages=num_stages, epochs_per_stage=epochs_per_stage)

pl_accs = train_ssl_unified('pseudo_label', model_pl, blocks_pl, labeled_loader_mt, unlabeled_loader_plain, test_loader_full, num_stages=num_stages, epochs_per_stage=epochs_per_stage)

mt_accs = train_ssl_unified('mean_teacher', student_mt, blocks_mt, labeled_loader_mt, unlabeled_loader_mt, test_loader_full, teacher=teacher_mt, teacher_blocks=teacher_blocks, num_stages=num_stages, epochs_per_stage=epochs_per_stage)



In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, total_epochs + 1), baseline_accs, label='Baseline (Labeled Only)', color='green')
plt.plot(range(1, total_epochs + 1), pl_accs, label='Pseudo Labeling', color='orange')
plt.plot(range(1, total_epochs + 1), mt_accs, label='Mean Teacher', color='blue')


plt.axvline(x=epochs_per_stage, color='red', linestyle=':', label='Stage 1 Unfreezing')
plt.axvline(x=epochs_per_stage*2, color='red', linestyle=':', label='Stage 2 Unfreezing')

plt.title('Strategy 2: SSL Comparison on 5% Labeled Data')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.legend()
plt.grid(True)
plt.show()